In [ ]:
!pip uninstall -y torchvision
!pip install -q transformers datasets accelerate sentencepiece scikit-learn anthropic

In [ ]:
import torch
import transformers
import datasets

print(torch.__version__)
print(transformers.__version__)
print(datasets.__version__)

In [ ]:
import pandas as pd
import numpy as np
import torch
import os

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from anthropic import Anthropic

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls "/content/drive/MyDrive/Capstone_AI/data"

In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/Capstone_AI/data/emotion_data.csv",
    encoding='cp949'
)

print(df.head())
print(df.columns)
print(df.shape)

In [ ]:
df = df[["사람문장1", "감정_대분류"]]

df = df.rename(
    columns={
        "사람문장1": "text",
        "감정_대분류": "emotion"
    }
)

df = df.dropna()

print(df.head())
print(df.shape)

In [ ]:
label_map = {
    "불안": 0,
    "분노": 1,
    "상처": 2,
    "슬픔": 3,
    "당황": 4,
    "기쁨": 5
}

id_to_label = {v: k for k, v in label_map.items()}

df = df[df["emotion"].isin(label_map.keys())].copy()
df["label"] = df["emotion"].map(label_map).astype(int)

print(df["emotion"].value_counts())
print(df["label"].value_counts())
print(df.shape)

In [ ]:
df = df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 2000), random_state=42)
).reset_index(drop=True)

print(df["emotion"].value_counts())
print(df["label"].value_counts())
print(df.shape)

In [ ]:
train_df, valid_df = train_test_split(
    df[["text", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_df, preserve_index=False)

print(train_dataset)
print(valid_dataset)

In [ ]:
model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6
)

print(model.config.num_labels)

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
valid_dataset = valid_dataset.rename_column("label", "labels")

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"]
)

print(train_dataset[0])

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_emotion_model",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
eval_result = trainer.evaluate()
print(eval_result)

In [ ]:
save_path = "/content/drive/MyDrive/Capstone_AI/model/final_emotion_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("모델 저장 완료:", save_path)

In [ ]:
print(predict_emotion("저녁을 같이 먹기로 했는데 다들 바빠서 결국 혼자 먹었다."))
print(predict_emotion("오늘 좋은 소식을 들어서 기분이 너무 좋았다."))
print(predict_emotion("계속 같은 실수를 해서 너무 화가 났다."))

In [ ]:
user_text = "저녁을 같이 먹기로 했는데 다들 바빠서 결국 혼자 먹었다."

emotion = predict_emotion(user_text)

print(emotion)

In [ ]:
user_role = "딸"
selected_mood = "슬픔"
user_text = "저녁을 같이 먹기로 했는데 다들 바빠서 결국 혼자 먹었다."

result = generate_messages(user_text, user_role, selected_mood)

print("역할:", result["role"])
print("선택 기분:", result["selected_mood"])
print("AI 감정:", result["ai_emotion"])

print("\n===== 사용자 팝업 =====")
print(result["user_message"])

print("\n===== 가족 팝업 =====")
print(result["family_message"])